# RQ2 Gate B1 — one completely fresh seed (T4×2)

Pre-registered screening run: train a fresh seed-6 Uniform-fixed trajectory from scratch, preserve epochs 10/50/100, extract SW and gradient geometry without updates, and apply the already-frozen Resource and Resource+SW predictors. Accuracy is never used for selection or tuning.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

## Rebuild development gates and freeze predictors

Input must contain the completed six-state quick trajectory diagnostic. The predictors are fitted only here, before seed 6 is trained or inspected.

In [ ]:
import importlib
import rq2_pairwise_external_gates, rq2_pairwise_surrogate_regret, rq2_quick_trajectory_diagnostic
rq2_pairwise_external_gates = importlib.reload(rq2_pairwise_external_gates)
rq2_pairwise_surrogate_regret = importlib.reload(rq2_pairwise_surrogate_regret)
DEVELOPMENT_GATE_DIR = Path('/kaggle/working/frozen-development-gates')
attached_frozen = sorted(Path('/kaggle/input').rglob('frozen_pairwise_predictors.json'))
attached_gate_a = sorted(Path('/kaggle/input').rglob('gate_a_summary.json'))
if len(attached_frozen) == 1 and len(attached_gate_a) == 1:
    import shutil
    DEVELOPMENT_GATE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(attached_frozen[0], DEVELOPMENT_GATE_DIR/'frozen_pairwise_predictors.json')
    shutil.copy2(attached_gate_a[0], DEVELOPMENT_GATE_DIR/'gate_a_summary.json')
    print('Using already-frozen Gate A/B0 artifacts:', attached_frozen[0].parent)
else:
    QUICK_ROOT = rq2_pairwise_surrogate_regret.find_quick_trajectory_root(
        Path('/kaggle/input'), '/kaggle/working/materialized-fresh-b1-development'
    )
    try:
        HT_ROOT = rq2_quick_trajectory_diagnostic.find_ht_development_root(
            Path('/kaggle/input'), '/kaggle/working/materialized-fresh-b1-ht'
        )
    except FileNotFoundError:
        HT_ROOT = None
    rq2_pairwise_external_gates.run_all_development_gates(
        QUICK_ROOT, DEVELOPMENT_GATE_DIR, ht_root=HT_ROOT
    )
FROZEN_PREDICTOR = DEVELOPMENT_GATE_DIR/'frozen_pairwise_predictors.json'
assert json.loads(FROZEN_PREDICTOR.read_text())['status'] == 'FROZEN_BEFORE_FRESH_STATES'
print('Development predictors frozen:', FROZEN_PREDICTOR)

## Freeze fresh protocol before training

Seed 6, Uniform-fixed anchors, 50+50 schedule, checkpoints 10/50/100. GPU 0 trains the single trajectory; GPU 1 is used later for parallel read-only extraction.

In [ ]:
import rq2_fresh_seed_gate_b1
rq2_fresh_seed_gate_b1 = importlib.reload(rq2_fresh_seed_gate_b1)
FRESH_ROOT = Path('/kaggle/working/fresh_seed_6')
FRESH_ROOT = rq2_fresh_seed_gate_b1.materialize_fresh_progress(
    Path('/kaggle/input'), FRESH_ROOT, '/kaggle/working/materialized-fresh-seed6-progress'
)
cifar_candidates = sorted({
    path.parent for path in Path('/kaggle/input').rglob('cifar-100-python')
    if path.is_dir() and all((path/name).is_file() for name in ('train','test','meta'))
})
assert len(cifar_candidates) <= 1, f'Multiple CIFAR-100 roots found: {cifar_candidates}'
DATASET_ROOT = cifar_candidates[0] if cifar_candidates else Path('/kaggle/working/cifar100-data')
print('CIFAR-100 source:', DATASET_ROOT, '(attached input)' if cifar_candidates else '(download/cache in working)')
CONFIG = rq2_fresh_seed_gate_b1.load_fresh_config(
    PROJECT_ROOT/'configs/kaggle_s1_extension_100.yaml', DATASET_ROOT
)
locked = {
 'status':'FROZEN_BEFORE_FRESH_TRAINING', 'seed':6,
 'trajectory':'fresh_uniform_fixed', 'anchors':[0.25,0.50,0.75,1.00],
 'schedule':'50+50_optimizer_scheduler_reset', 'checkpoints':[10,50,100],
 'frozen_predictor_sha256': __import__('hashlib').sha256(FROZEN_PREDICTOR.read_bytes()).hexdigest(),
 'accuracy_used_for_tuning':False, 'test_used':False, 'gate_c_authorized':False,
 'git_commit':GIT_COMMIT
}
FRESH_ROOT.mkdir(parents=True, exist_ok=True)
protocol_file = FRESH_ROOT/'frozen_fresh_protocol.json'
if protocol_file.exists():
    previous = json.loads(protocol_file.read_text())
    scientific_keys = ['status','seed','trajectory','anchors','schedule','checkpoints','frozen_predictor_sha256','accuracy_used_for_tuning','test_used','gate_c_authorized']
    assert all(previous.get(key) == locked.get(key) for key in scientific_keys), 'Frozen scientific protocol changed across resume'
    previous.setdefault('resume_code_commits', [])
    if GIT_COMMIT not in previous['resume_code_commits']: previous['resume_code_commits'].append(GIT_COMMIT)
    protocol_file.write_text(json.dumps(previous, indent=2)+'\n')
    locked = previous
else:
    protocol_file.write_text(json.dumps(locked, indent=2)+'\n')
print(json.dumps(locked, indent=2))

## Train the fresh trajectory from scratch (exactly resumable)

In [ ]:
started = time.perf_counter()
rq2_fresh_seed_gate_b1.train_fresh_trajectory(CONFIG, FRESH_ROOT)
print(f'Fresh trajectory training completed/resumed in {(time.perf_counter()-started)/3600:.2f} hours')
for epoch in (10,50,100):
    path = FRESH_ROOT/'checkpoints'/f'epoch_{epoch:03d}.pt'
    assert path.is_file(); print(path, f'{path.stat().st_size/2**20:.1f} MiB')

## Extract three fresh states read-only on T4×2

Epochs 10 and 50 start concurrently; epoch 100 starts on the first released GPU. Gradients are evaluation targets only.

In [ ]:
import scripts.run_fresh_gate_b1 as fresh_runner
fresh_runner = importlib.reload(fresh_runner)
workers = fresh_runner.run_workers(
    FRESH_ROOT, DATASET_ROOT, DEVELOPMENT_GATE_DIR/'gate_a_summary.json', gpu_ids=(0,1)
)
print('Completed workers:', workers)

## Apply frozen predictors and compute exact Gate B1 variances

In [ ]:
summary = rq2_fresh_seed_gate_b1.merge_and_evaluate_fresh_states(
    FRESH_ROOT, workers, FROZEN_PREDICTOR
)
print(json.dumps(summary, indent=2))
import pandas as pd
from IPython.display import display
display(pd.read_csv(FRESH_ROOT/'evaluation/resource_vs_hybrid.csv'))

## Validate and export

In [ ]:
required = [
 'checkpoints/epoch_010.pt','checkpoints/epoch_050.pt','checkpoints/epoch_100.pt',
 'state_metadata.json','pair_structure.csv',
 'sw_matrices/epoch_010.npy','sw_matrices/epoch_050.npy','sw_matrices/epoch_100.npy',
 'gradient_grams/epoch_010.npy','gradient_grams/epoch_050.npy','gradient_grams/epoch_100.npy',
 'evaluation/resource_vs_hybrid.csv','evaluation/gate_b1_summary.json'
]
missing = [name for name in required if not (FRESH_ROOT/name).is_file() or (FRESH_ROOT/name).stat().st_size == 0]
assert not missing, f'Missing fresh Gate B1 artifacts: {missing}'
saved = json.loads((FRESH_ROOT/'evaluation/gate_b1_summary.json').read_text())
assert saved['predictors_refit_on_fresh_states'] is False
assert saved['gate_c_authorized'] is False and saved['one_seed_screening_only'] is True
bundle_path = Path('/kaggle/working/rq2-fresh-seed6-gate-b1.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in FRESH_ROOT.rglob('*'):
        if path.is_file(): bundle.write(path, Path('fresh_seed_6')/path.relative_to(FRESH_ROOT))
    for path in DEVELOPMENT_GATE_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, Path('frozen_development_gates')/path.relative_to(DEVELOPMENT_GATE_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**30:.2f} GiB')
bundle_path